# Cloud benchmark setup

This notebook sorts the Maxar images into **validation and test**, gives every crop a short name, saves each crop with **OCM's mask ready for you to fix**, and later scores **OmniCloudMask plus the classical methods** the same way.

**It never changes your `to_fix/` folder.** New files go in `clean_dataset/`.

**How to read a name:** `test_03_c1` = **test** group, image **3**, crop **1**.

Groups: `test`, `val` (validation), `set_aside`. Where and when each image was taken, and how cloudy it is, are in `label_tracker.csv`.

The original Maxar name is always in `image_lookup.csv` and `label_tracker.csv`.

## Settings

In [ ]:
# Works in Colab and on your PC. On your PC it looks for Google Drive for desktop
# automatically; if it can't find it, paste your folder path into LOCAL_PROJECT_DIR.
LOCAL_PROJECT_DIR = r"G:\My Drive\cloud_project"

import os, string
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/ursa space 1C data"
except ImportError:
    found = [f"{d}:/My Drive/ursa space 1C data" for d in string.ascii_uppercase
             if os.path.isdir(f"{d}:/My Drive/ursa space 1C data")]
    PROJECT_DIR = LOCAL_PROJECT_DIR or (found[0] if found else "")
    assert PROJECT_DIR, "Couldn't find your Drive folder: set LOCAL_PROJECT_DIR above."
print("Using", PROJECT_DIR)
INVENTORY = PROJECT_DIR + "/inventory_NWT-Canada-Aug-23.csv"
OUT = PROJECT_DIR + "/clean_dataset"

GROUPS_TO_MAKE = ["test", "val"]
CROPS_PER_IMAGE = 3
CROP_PX = 2048

THIN_IS_CLOUD = True     # decided: thin cloud counts as cloud
SMOKE_IS_CLOUD = False   # decided: smoke is NOT cloud (label it 4 so smoke errors can be studied)
RUN_FINAL_TEST = False   # turn on once only, after settings are locked

In [ ]:
%pip install -q omnicloudmask rasterio

In [ ]:
import torch
os.makedirs(OUT, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (OCM will be slower)")

## Helper code

In [ ]:
# ===============================================================
# All the helper code. You don't need to edit this cell.
# ===============================================================
import os, re, shutil, tempfile, json
import numpy as np, pandas as pd, cv2, rasterio
from rasterio.enums import Resampling
from rasterio.windows import Window
from scipy import ndimage as ndi

# Mask values
CLEAR, THICK, THIN, SHADOW, SMOKE, UNSURE, NODATA = 0, 1, 2, 3, 4, 5, 255
MASK_NAME = "mask_ocm_assisted.tif"   # the mask you edit
VALID_MASK_VALUES = (CLEAR, THICK, THIN, SHADOW, SMOKE, UNSURE, NODATA)
CAPTURES = {"104001008B463900": ("yellowknife", "aug08"), "104001008892B200": ("yellowknife", "aug15"),
            "104001008A8DAA00": ("kelowna", "aug21"), "1040010088356D00": ("kelowna", "aug21")}

# ---------- 1. Who goes in which group ---------------------------
def quad_xy(quadkey):
    """Turn a Maxar location code into a column and row on a map grid."""
    x = y = 0
    for ch in str(quadkey):
        d = int(ch); x = x * 2 + (d & 1); y = y * 2 + (d >> 1)
    return x, y

def cover_word(pct):
    return "clear" if pct < 5 else ("partly" if pct <= 50 else "cloudy")

def build_split(inv):
    """One row per image: its group, reason, and easy name.
    kelowna: west 2 columns = test, next column = set aside (a gap so the same
    clouds aren't in both), the rest = val.
    yellowknife aug15 = all validation. yellowknife aug08 (clear days): west
    column = test, the rest = val."""
    r = inv.copy()
    p = r["scene_id"].str.extract(r"^(?P<zone>\d+)_(?P<quad>\d+)_(?P<pass_id>\w+)$")
    r = pd.concat([r, p], axis=1)
    bad = sorted(set(r["pass_id"]) - set(CAPTURES))
    if bad: raise ValueError(f"Unknown satellite pass: {bad}")
    r["place"] = r["pass_id"].map(lambda c: CAPTURES[c][0])
    r["date"] = r["pass_id"].map(lambda c: CAPTURES[c][1])
    xy = r["quad"].map(quad_xy)
    r["x"], r["y"] = xy.str[0], xy.str[1]
    r["col"] = r["x"] - r.groupby("place")["x"].transform("min")
    r["row"] = r["y"] - r.groupby("place")["y"].transform("min")
    def rule(a):
        if a["place"] == "kelowna":
            if a["col"] <= 1: return "test"
            return "set_aside" if a["col"] == 2 else "val"
        if a["date"] == "aug15": return "val"
        return "test" if a["col"] == 0 else "val"
    r["group"] = r.apply(rule, axis=1)
    # Images that used to be train are numbered after the original val images,
    # so val_01 to val_16 keep their existing names and crop folders.
    r["added_val"] = (r["group"] == "val") & ~((r["place"] == "yellowknife") & (r["date"] == "aug15"))
    r["cover"] = r["tile:clouds_percent"].map(cover_word)
    r = r.sort_values(["place", "date", "row", "col", "pass_id"]).reset_index(drop=True)
    # Short names: the group plus a number, e.g. test_01. Details live in the lookup sheet.
    r = r.sort_values(["group", "added_val", "place", "date", "row", "col", "pass_id"]).reset_index(drop=True)
    r["easy_tile"] = r["group"] + "_" + (r.groupby("group").cumcount() + 1).map("{:02d}".format)
    return r

def check_split(plan):
    """Stop if the same location/time could end up in two groups."""
    used = plan[plan["group"] != "set_aside"]
    k = used[used["place"] == "kelowna"]
    assert k.groupby(["quad"])["group"].nunique().max() == 1, "same place, same day, two groups"
    te, va = k[k["group"] == "test"], k[k["group"] == "val"]
    assert va["col"].min() - te["col"].max() >= 2, "no gap between Kelowna test and val"
    assert set(plan["group"]) == {"test", "val", "set_aside"}, "unexpected group"
    assert not (set(used.loc[used["group"] == "test", "date"]) & {"aug15"})
    assert plan["easy_tile"].is_unique
    return True


# Colour style QGIS loads automatically for any mask with the same file name
QML = """<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>
<qgis version="3.28" styleCategories="Symbology">
  <pipe>
    <rasterrenderer type="paletted" band="1" opacity="0.5" alphaBand="-1" nodataColor="">
      <colorPalette>
        <paletteEntry value="0" color="#000000" alpha="0" label="0 Clear"/>
        <paletteEntry value="1" color="#ff00ff" alpha="255" label="1 Thick cloud"/>
        <paletteEntry value="2" color="#00e5ff" alpha="255" label="2 Thin cloud"/>
        <paletteEntry value="3" color="#ffd400" alpha="255" label="3 Cloud shadow"/>
        <paletteEntry value="4" color="#ff6a00" alpha="255" label="4 Smoke"/>
        <paletteEntry value="5" color="#00ff3c" alpha="255" label="5 Unsure"/>
      </colorPalette>
    </rasterrenderer>
  </pipe>
</qgis>
"""

# ---------- 2. Making crops --------------------------------------
def nodata_mask(rgb): return (rgb == 0).all(axis=0)

def run_ocm_tile(red, green, nir=None):
    """OmniCloudMask on a whole low-resolution tile (zeros replace NIR by default)."""
    from omnicloudmask import predict_from_array
    nod = (red == 0) & (green == 0)
    arr = np.stack([red, green, np.zeros_like(red) if nir is None else nir]).astype(np.float32)
    if nod.any() and (~nod).any():
        for b in range(2 if nir is None else 3):
            arr[b][nod] = np.median(arr[b][~nod])
    ps = max(32, min(1000, *arr.shape[1:]) // 32 * 32)
    m = np.asarray(predict_from_array(arr, patch_size=ps, patch_overlap=min(300, ps // 3), apply_no_data_mask=False))
    m = m.reshape(m.shape[-2:]).astype(np.uint8); m[nod] = NODATA
    return m

def read_ocm_rgb(src, work_res_m=10):
    f = max(1.0, work_res_m / abs(src.transform.a))
    h, w = max(1, round(src.height / f)), max(1, round(src.width / f))
    c = src.read([1, 2], out_shape=(2, h, w), resampling=Resampling.average)
    return run_ocm_tile(c[0], c[1]), src.width / w, src.height / h

def pick_windows(ocm, sx, sy, width, height, crop, n, rng):
    """Choose a useful mix: most mixed cloud/clear, then cloudiest, then clearest."""
    rows = []
    for r in range(0, height - crop + 1, crop):
        for c in range(0, width - crop + 1, crop):
            a = ocm[int(r / sy):max(int(r / sy) + 1, int(np.ceil((r + crop) / sy))),
                    int(c / sx):max(int(c / sx) + 1, int(np.ceil((c + crop) / sx)))]
            v = a != NODATA
            if v.mean() > 0.98:
                rows.append((c, r, float(np.isin(a[v], [THICK, THIN]).mean())))
    if not rows: return []
    s = pd.DataFrame(rows, columns=["col", "row", "ocm_cloud"])
    s["u"] = rng.random(len(s))
    orders = [("mixed", s.assign(k=(s["ocm_cloud"] - .5).abs()).sort_values(["k", "u"])),
              ("cloudiest", s.sort_values(["ocm_cloud", "u"], ascending=[False, True])),
              ("clearest", s.sort_values(["ocm_cloud", "u"]))]
    out, used = [], set()
    while len(out) < min(n, len(s)):
        for why, o in orders:
            for i, a in o.iterrows():
                if i not in used:
                    used.add(i); out.append((int(a["col"]), int(a["row"]), why, float(a["ocm_cloud"]))); break
            if len(out) >= min(n, len(s)): break
    return out

def write_tif(path, arr, crs, transform, nodata=None):
    arr = arr if arr.ndim == 3 else arr[None]
    prof = dict(driver="GTiff", height=arr.shape[1], width=arr.shape[2], count=arr.shape[0], dtype=arr.dtype,
                crs=crs, transform=transform, compress="deflate")
    if nodata is not None: prof["nodata"] = nodata
    if arr.shape[0] == 3: prof["photometric"] = "RGB"
    with rasterio.open(path, "w", **prof) as d: d.write(arr)

def make_tile_crops(row, out_root, crop_px=2048, n=3, seed=0):
    rng = np.random.default_rng(seed + int(row["quad"]) % 100000)
    out = []
    with rasterio.open(row["visual_url"]) as src:
        ocm, sx, sy = read_ocm_rgb(src)
        for k, (c, r, why, frac) in enumerate(pick_windows(ocm, sx, sy, src.width, src.height, crop_px, n * 4, rng)):
            if len(out) >= n: break
            win = Window(c, r, crop_px, crop_px)
            rgb = src.read([1, 2, 3], window=win)
            if nodata_mask(rgb).mean() > 0.02: continue
            name = f"{row['easy_tile']}_c{len(out) + 1}"
            folder = os.path.join(out_root, "ocm_assisted", row["group"], name)
            t = src.window_transform(win)
            r0, c0 = int(r / sy), int(c / sx)
            sub = ocm[r0:int(np.ceil((r + crop_px) / sy)), c0:int(np.ceil((c + crop_px) / sx))]
            d = cv2.resize(sub, (crop_px, crop_px), interpolation=cv2.INTER_NEAREST).astype(np.uint8)
            d[nodata_mask(rgb)] = NODATA
            # untouched copy of OCM's answer, used later to score OCM
            pred_dir = os.path.join(out_root, "_ocm_predictions"); os.makedirs(pred_dir, exist_ok=True)
            write_tif(os.path.join(pred_dir, name + ".tif"), d, src.crs, t, nodata=NODATA)
            # your workspace: image + OCM mask together, ready to overlay and fix
            os.makedirs(folder, exist_ok=True)
            write_tif(os.path.join(folder, "image.tif"), rgb, src.crs, t)
            if not os.path.exists(os.path.join(folder, MASK_NAME)):   # never overwrite your edits
                write_tif(os.path.join(folder, MASK_NAME), d, src.crs, t, nodata=NODATA)
            with open(os.path.join(folder, MASK_NAME.replace(".tif", ".qml")), "w") as fh: fh.write(QML)
            out.append({"crop_name": name, "group": row["group"], "easy_tile": row["easy_tile"],
                        "original_tile": row["scene_id"], "place": row["place"], "date": row["date"],
                        "maxar_tile_cloud_pct": row["tile:clouds_percent"], "why_chosen": why,
                        "ocm_cloud": round(frac, 3), "col_px": c, "row_px": r, "folder": folder})
    return out

def label_priority(name, group):
    c = int(name.rsplit("_c", 1)[1])
    if group == "test" and c <= 2: return 1
    if group == "val" and c == 1: return 2
    return 3

# ---------- 3. Classical methods ---------------------------------
# Each returns a score per pixel; higher = more cloud-like.
def to01(rgb): return rgb.astype(np.float32) / 255

def whiteness_score(rgb):
    x = to01(rgb); mx, mn = x.max(0), x.min(0)
    return x.mean(0) * (1 - (mx - mn) / np.maximum(mx, 1e-6))

def dark_channel(rgb, window=15):
    return cv2.erode(to01(rgb).min(0), np.ones((window, window), np.uint8))

def dark_channel_score(rgb): return dark_channel(rgb)

def hot_score(rgb, median=17):
    """Blue - 0.5 x Red, smoothed because HOT is noisy at very high resolution."""
    x = to01(rgb)
    u8 = np.clip((x[2] - 0.5 * x[0] + 0.5) / 1.5 * 255, 0, 255).astype(np.uint8)
    return cv2.medianBlur(u8, median).astype(np.float32) / 255 * 1.5 - 0.5

CLASSICAL = {"whiteness": whiteness_score, "dark_channel": dark_channel_score, "hot": hot_score}

# ---------- 4. Scoring --------------------------------------------
def reference_to_binary(mask, thin_is_cloud=True, smoke_is_cloud=False):
    """Returns (truth, usable). Unsure and outside-image pixels aren't scored."""
    pos = [THICK] + ([THIN] if thin_is_cloud else []) + ([SMOKE] if smoke_is_cloud else [])
    return np.isin(mask, pos), ~np.isin(mask, [UNSURE, NODATA])

def counts(pred, truth, usable):
    p, t = pred[usable], truth[usable]
    return dict(tp=int((p & t).sum()), fp=int((p & ~t).sum()), fn=int((~p & t).sum()), tn=int((~p & ~t).sum()))

def metrics(c):
    tp, fp, fn = c["tp"], c["fp"], c["fn"]
    d = lambda a, b: a / b if b else float("nan")
    return dict(iou=d(tp, tp + fp + fn), precision=d(tp, tp + fp), recall=d(tp, tp + fn), f1=d(2 * tp, 2 * tp + fp + fn))

def finished_crops(tracker, group):
    out = []
    for _, r in tracker[tracker["group"] == group].iterrows():
        if r.get("fixed", False) and os.path.exists(os.path.join(r["folder"], MASK_NAME)):
            out.append((r["crop_name"], r["folder"]))
    return out

def read_pair(folder):
    with rasterio.open(os.path.join(folder, "image.tif")) as a: rgb = a.read([1, 2, 3])
    with rasterio.open(os.path.join(folder, MASK_NAME)) as b: mask = b.read(1)
    if mask.shape != rgb.shape[1:]: raise ValueError(f"{folder}: mask and image sizes differ")
    bad = set(np.unique(mask).tolist()) - set(VALID_MASK_VALUES)
    if bad: raise ValueError(f"{folder}: unexpected mask values {sorted(bad)}")
    return rgb, mask

def score_arrays(method, rgb):
    return CLASSICAL[method](rgb)

def tune_thresholds(crops, thin=True, smoke=False, steps=41):
    """Try many thresholds on validation crops only; keep the best IoU per method."""
    grid = np.linspace(0, 1, steps)
    best = {}
    if not crops: raise ValueError("No finished masks yet")
    data = [read_pair(f) for _, f in crops]
    for m in CLASSICAL:
        tot = np.zeros((steps, 3), np.int64); lo, hi = np.inf, -np.inf; sc = []
        for rgb, mk in data:
            s = score_arrays(m, rgb); sc.append(s); lo, hi = min(lo, float(s.min())), max(hi, float(s.max()))
        ths = lo + grid * (hi - lo)
        for s, (rgb, mk) in zip(sc, data):
            t, u = reference_to_binary(mk, thin, smoke)
            for i, th in enumerate(ths):
                c = counts(s >= th, t, u); tot[i] += [c["tp"], c["fp"], c["fn"]]
        iou = tot[:, 0] / np.maximum(tot.sum(1), 1)
        best[m] = float(ths[int(iou.argmax())])
    return best

def evaluate(crops, thresholds, thin=True, smoke=False):
    """Pooled pixel scores per method, plus one row per crop for failure analysis."""
    tot, per = {}, []
    for name, f in crops:
        rgb, mk = read_pair(f); t, u = reference_to_binary(mk, thin, smoke)
        preds = {m: score_arrays(m, rgb) >= th for m, th in thresholds.items()}
        p = os.path.join(OUT, "_ocm_predictions", name + ".tif")
        if os.path.exists(p):
            with rasterio.open(p) as d: preds["ocm_rgb"] = np.isin(d.read(1), [THICK, THIN])
        for m, pr in preds.items():
            c = counts(pr, t, u); per.append({"crop": name, "method": m, **metrics(c)})
            q = tot.setdefault(m, dict(tp=0, fp=0, fn=0, tn=0))
            for k in q: q[k] += c[k]
    table = pd.DataFrame([{"method": m, **metrics(c)} for m, c in tot.items()])
    if len(table): table = table.sort_values("iou", ascending=False).reset_index(drop=True)
    return table, pd.DataFrame(per)

def cvat_png_to_mask(png_path, image_tif, out_tif):
    """Convert a CVAT colour PNG (clear/cloud/uncertain) to mask.tif."""
    from PIL import Image
    a = np.array(Image.open(png_path).convert("RGB")).astype(int)
    pal = {(52, 209, 183): CLEAR, (240, 120, 240): THICK, (184, 61, 245): UNSURE, (0, 0, 0): UNSURE}
    out = np.full(a.shape[:2], UNSURE, np.uint8)
    for col, v in pal.items(): out[(a == col).all(-1)] = v
    with rasterio.open(image_tif) as s:
        if (s.height, s.width) != out.shape: raise ValueError("PNG and image sizes differ")
        write_tif(out_tif, out, s.crs, s.transform, nodata=NODATA)


## Step 1: Sort every image into a group

- **Set aside:** a one-image-wide strip of Kelowna, so the same clouds don't land in both validation and test.
- **Validation:** all Yellowknife images from Aug 15, Kelowna's east side, and most clear Yellowknife images. Use these to choose method settings.
- **Test:** Kelowna, west side, plus four clear Yellowknife images. Final scores come from here.

In [ ]:
inv = pd.read_csv(INVENTORY)
plan = build_split(inv)
check_split(plan)
plan[["easy_tile", "group", "place", "date", "tile:clouds_percent", "scene_id"]].rename(
    columns={"tile:clouds_percent": "maxar_cloud_pct", "scene_id": "original_name"}
).to_csv(OUT + "/image_lookup.csv", index=False)
group_n = plan["group"].value_counts()
print(f"{len(plan)} images total = {group_n['test']} test, {group_n['val']} val, {group_n['set_aside']} set_aside")
display(plan.groupby(["group", "place", "date"]).size().rename("images").reset_index())

## Step 2: Make the crops

Roughly 1-3 minutes per image. If it stops, run this cell again: finished images are skipped.

In [ ]:
tracker_path = OUT + "/label_tracker.csv"
tracker = pd.read_csv(tracker_path) if os.path.exists(tracker_path) else pd.DataFrame()
done = set(tracker["original_tile"]) if len(tracker) else set()
skip_path = OUT + "/skipped_images.csv"
skipped = pd.read_csv(skip_path) if os.path.exists(skip_path) else pd.DataFrame(columns=["original_tile", "reason"])
todo = plan[plan["group"].isin(GROUPS_TO_MAKE) & ~plan["scene_id"].isin(done | set(skipped["original_tile"]))]
print(len(todo), "images to process")
for i, (_, r) in enumerate(todo.iterrows(), 1):
    print(f"[{i}/{len(todo)}] {r['easy_tile']}")
    try:
        rows = make_tile_crops(r, OUT, CROP_PX, CROPS_PER_IMAGE)
        why = "" if rows else "no clean crop fits"
    except Exception as e:
        rows, why = [], repr(e)
    if rows:
        new = pd.DataFrame(rows)
        new["priority"] = [label_priority(n, g) for n, g in zip(new["crop_name"], new["group"])]
        new["fixed"] = False; new["notes"] = ""
        tracker = pd.concat([tracker, new], ignore_index=True)
        tracker.to_csv(tracker_path, index=False)
    else:
        print("   skipped:", why)
        skipped = pd.concat([skipped, pd.DataFrame([{"original_tile": r["scene_id"], "reason": why}])], ignore_index=True)
        skipped.to_csv(skip_path, index=False)
print(len(tracker), "crops ready")
if len(tracker): display(tracker.groupby(["group", "priority"]).size().rename("crops").reset_index())

## Step 3: Fix the masks

Your crops are in **`clean_dataset/ocm_assisted/`**, split into `test/` and `val/`. Each crop folder has:

| File | What it is |
|---|---|
| `image.tif` | The sharp satellite image (bottom layer) |
| `mask_ocm_assisted.tif` | OCM's mask: **the one you edit** |
| `mask_ocm_assisted.qml` | Shows the mask in colour, 50% see-through |

**In QGIS:** drag in `image.tif`, then `mask_ocm_assisted.tif` on top. Fix it with Serval and save.

| Value | Meaning |
|---|---|
| 0 | Clear |
| 1 | Thick cloud |
| 2 | Thin cloud |
| 3 | Cloud shadow |
| 4 | Smoke |
| 5 | Unsure (left out of scoring) |
| 255 | Outside the image |

When a crop is finished, put `TRUE` in its `fixed` column in `label_tracker.csv`. Only finished crops are scored.

In [ ]:
tracker = pd.read_csv(tracker_path)
if "fixed" not in tracker: tracker["fixed"] = False
tracker["fixed"] = tracker["fixed"].fillna(False).astype(bool)
tracker.to_csv(tracker_path, index=False)
display(tracker.groupby(["group"])["fixed"].agg(finished="sum", total="size").reset_index())

## Step 4: Choose method settings on validation

In [ ]:
val = finished_crops(tracker, "val")
print(len(val), "finished validation crops")
thresholds = tune_thresholds(val, THIN_IS_CLOUD, SMOKE_IS_CLOUD)
json.dump(thresholds, open(OUT + "/locked_settings.json", "w"), indent=2)
print(thresholds)
val_table, _ = evaluate(val, thresholds, THIN_IS_CLOUD, SMOKE_IS_CLOUD)
val_table.round(3)

## Step 5: Final scores on test

Only run this after labels and settings are final. Changing methods after seeing these scores makes the test unfair. OCM uses red, green and zeros for NIR here.

In [ ]:
assert RUN_FINAL_TEST, "Set RUN_FINAL_TEST = True when settings are locked."
thresholds = json.load(open(OUT + "/locked_settings.json"))
test = finished_crops(tracker, "test")
results, per_crop = evaluate(test, thresholds, THIN_IS_CLOUD, SMOKE_IS_CLOUD)
results.to_csv(OUT + "/final_results.csv", index=False)
per_crop.to_csv(OUT + "/final_results_per_crop.csv", index=False)
print(len(test), "test crops")
results.round(3)